# 2 — Rules and inference

*The SemOps Manual — Chapter 11*

SHACL-AF rules derive triples before validation, so a report can depend on data
that was never asserted. This notebook runs each rule form, then reproduces the
four ways a rule silently does nothing.

---

## Prerequisites

This notebook is **self-contained**: it writes its own fixtures into a temporary
directory, so nothing needs to exist on disk beforehand. What it does need:

```bash
pip install ontology-quality-suite shacl
```

The next cell checks what is available and prints exactly what is missing.
Cells that need a tool you do not have will say so and skip, rather than
failing with a stack trace.

In [ ]:
import json, os, shutil, subprocess, sys, tempfile, textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="semops-nb-"))
print("working directory:", WORK)


def have(mod):
    try:
        __import__(mod)
        return True
    except ImportError:
        return False


HAVE_SUITE = have("ontology_suite")
HAVE_SHACL = have("shacl")
SHACL_CLI = shutil.which("shacl")

print("ontology-quality-suite:", "yes" if HAVE_SUITE else "NO  (pip install ontology-quality-suite)")
print("shacl (python)       :", "yes" if HAVE_SHACL else "NO  (pip install shacl)")
print("shacl (cli)          :", SHACL_CLI or "not on PATH")

if HAVE_SHACL:
    import shacl as shacl_py
    print("shacl version        :", getattr(shacl_py, "__version__", "unknown"))


def write(name, text):
    """Write a fixture into the working directory and return its path."""
    p = WORK / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(text).lstrip(), encoding="utf-8")
    return p


def suite(*args):
    """Run the ontology suite CLI and show its output."""
    if not HAVE_SUITE:
        print("skipped: ontology-quality-suite is not installed")
        return None
    r = subprocess.run([sys.executable, "-m", "ontology_suite", *map(str, args)],
                       capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[:2000])
    return r

In [ ]:
# Rules need the shacl engine, not the ontology suite -- the suite has no
# --advanced flag. The CLI is used here because the Python binding only gained
# inference="rules" at 0.1.7; earlier versions accept "none" and "rdfs" only.
def rules(data, shapes, *extra, advanced=True):
    if not SHACL_CLI:
        print("skipped: the shacl CLI is not on PATH (pip install shacl)")
        return None
    cmd = [SHACL_CLI, "-d", str(data), "-s", str(shapes), *map(str, extra)]
    if advanced:
        cmd.append("--advanced")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout or r.stderr).strip()[:1500] or "(no output)")
    print("exit:", r.returncode, " [0=conforms 1=violations 2=could not validate]")
    return r

## Triple rules

A triple rule builds triples from three node expressions, evaluated per focus
node. Here: every `Person` becomes an `Agent`, and each `knows` value produces
a `contact`.

To prove the rule fired we validate the **derived** data — `AgentShape` targets
a class that only exists because a rule created it.

In [ ]:
shapes = write("triple.ttl", """
    @prefix sh:  <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix ex:  <http://example.org/> .

    ex:PersonShape a sh:NodeShape ;
        sh:targetClass ex:Person ;
        sh:rule [ a sh:TripleRule ;
            sh:subject sh:this ; sh:predicate rdf:type ; sh:object ex:Agent ] ;
        sh:rule [ a sh:TripleRule ;
            sh:subject sh:this ; sh:predicate ex:contact ;
            sh:object [ sh:path ex:knows ] ] .

    ex:AgentShape a sh:NodeShape ;
        sh:targetClass ex:Agent ;
        sh:property [ sh:path ex:contact ; sh:minCount 1 ;
                      sh:message "Agent has no contact" ] .
""")

data = write("people.ttl", """
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:knows ex:bob, ex:carol .
    ex:dave  a ex:Person .
""")

print("--- rules OFF: ex:Agent does not exist, so nothing is targeted ---")
rules(data, shapes, advanced=False)
print()
print("--- rules ON: alice gains two contacts and passes; dave gains none ---")
rules(data, shapes)

## SPARQL rules

`sh:construct` runs a CONSTRUCT with `$this` pre-bound. This is where anything
computed lives. The shape then asserts the derived value is exactly 12, so a
conforming result proves the arithmetic ran.

In [ ]:
shapes = write("sparql.ttl", '''
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix ex: <http://example.org/> .

    ex:BoxShape a sh:NodeShape ; sh:targetClass ex:Box ;
        sh:rule [ a sh:SPARQLRule ; sh:construct """
            PREFIX ex: <http://example.org/>
            CONSTRUCT { $this ex:area ?a }
            WHERE { $this ex:width ?w ; ex:height ?h . BIND(?w * ?h AS ?a) }""" ] ;
        sh:property [ sh:path ex:area ; sh:minCount 1 ; sh:hasValue 12 ;
                      sh:message "area must be 12" ] .
''')
data = write("box.ttl", '@prefix ex: <http://example.org/> . ex:b a ex:Box ; ex:width 3 ; ex:height 4 .')
rules(data, shapes)

---

## Four ways to get a wrong answer

Each of these produces **silence**, not an error. That is what makes them worth
meeting deliberately, once, in a notebook.

### 1. A rule under `sh:property` never fires

A rule fires on its shape's targets. A nested property shape has no target, so
the rule reads naturally and does nothing at all.

In [ ]:
shapes = write("nested.ttl", """
    @prefix sh:  <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix ex:  <http://example.org/> .

    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
        sh:property [ sh:path ex:name ;
            sh:rule [ a sh:TripleRule ;
                sh:subject sh:this ; sh:predicate rdf:type ; sh:object ex:Tagged ] ] .

    ex:Check a sh:NodeShape ; sh:targetClass ex:Tagged ;
        sh:property [ sh:path ex:nope ; sh:minCount 1 ] .
""")
data = write("nested-data.ttl", '@prefix ex: <http://example.org/> . ex:p a ex:Person ; ex:name "x" .')
rules(data, shapes)
print("\n^ conforms, because ex:Tagged was never created. Move the rule up to the node shape.")

### 2. One pass, so a transitive rule does not close

SHACL-AF defines a single iteration. On the chain `a -> b -> c -> d`, deriving
`sub` from two hops of `sub` gets `a` as far as `c` and stops.

In [ ]:
shapes = write("trans.ttl", '''
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix ex: <http://example.org/> .

    ex:S a sh:NodeShape ; sh:targetClass ex:T ;
        sh:rule [ a sh:SPARQLRule ; sh:construct """
            PREFIX ex: <http://example.org/>
            CONSTRUCT { $this ex:sub ?z }
            WHERE { $this ex:sub ?y . ?y ex:sub ?z }""" ] ;
        sh:property [ sh:path ex:sub ;
            sh:qualifiedValueShape [ sh:hasValue ex:d ] ; sh:qualifiedMinCount 1 ;
            sh:message "should reach ex:d transitively" ] .
''')
data = write("chain.ttl", """
    @prefix ex: <http://example.org/> .
    ex:a a ex:T ; ex:sub ex:b . ex:b a ex:T ; ex:sub ex:c . ex:c a ex:T ; ex:sub ex:d .
""")

print("--- one pass: a does not reach d ---")
rules(data, shapes)
print()
print("--- --iterate-rules 10: it does ---")
rules(data, shapes, "--iterate-rules", 10)

`--iterate-rules` is **outside the specification**. Another SHACL processor will
not reproduce these results, so treat using it as a documented decision rather
than a default.

### 3. Rules at the same `sh:order` cannot see each other

Two rules at the default order 0 each see the graph as it was before either ran,
so one cannot consume what the other produces.

In [ ]:
shapes = write("order.ttl", """
    @prefix sh:  <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix ex:  <http://example.org/> .

    ex:A a sh:NodeShape ; sh:targetClass ex:Person ;
        sh:rule [ a sh:TripleRule ; sh:subject sh:this ;
                  sh:predicate rdf:type ; sh:object ex:Step1 ] .
    ex:B a sh:NodeShape ; sh:targetClass ex:Step1 ;
        sh:rule [ a sh:TripleRule ; sh:subject sh:this ;
                  sh:predicate rdf:type ; sh:object ex:Step2 ] .
    ex:C a sh:NodeShape ; sh:targetClass ex:Step2 ;
        sh:property [ sh:path ex:nope ; sh:minCount 1 ; sh:message "reached Step2" ] .
""")
data = write("order-data.ttl", '@prefix ex: <http://example.org/> . ex:p a ex:Person .')

print("--- same order: Step2 never reached ---")
rules(data, shapes)
print()
print("--- iterated: it is ---")
rules(data, shapes, "--iterate-rules", 5)
print("\nThe principled fix is sh:order on the consuming shape, not iteration.")

### 4. Negation is not monotonic

Rules only ever add triples, so a conclusion drawn from *absence* outlives the
absence. Round 1 marks a person `Unnamed`; round 2 gives them a name; the mark
stays, and is now simply false.

If a rule tests for absence, either keep to a single pass or make sure nothing
later supplies what it tested for.

---

## Unsupported features fail loudly

The failure mode that matters most in a gate is the one that reports success
when it could not check. Here a SHACL **function** — deliberately not
implemented — is used in a rule.

In [ ]:
shapes = write("fn.ttl", """
    @prefix sh:  <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    @prefix ex:  <http://example.org/> .

    ex:double a sh:SPARQLFunction ;
        sh:parameter [ sh:path ex:n ; sh:datatype xsd:integer ] ;
        sh:returnType xsd:integer ;
        sh:select "SELECT ($n * 2 AS ?result) WHERE {}" .

    ex:S a sh:NodeShape ; sh:targetClass ex:Thing ;
        sh:rule [ a sh:TripleRule ; sh:subject sh:this ;
            sh:predicate ex:twice ; sh:object [ ex:double ( 21 ) ] ] .
""")
data = write("fn-data.ttl", '@prefix ex: <http://example.org/> . ex:t a ex:Thing .')
rules(data, shapes)
print("\n^ exit 2, not 0. 'Could not validate' is a different outcome from 'valid'.")

---

## What to take away

- Rules are **opt-in** (`--advanced`). The same data and shapes give a different
  report with and without them, which is why they are not automatic.
- **Validate the derived triples.** A rule that stops firing is invisible unless
  something downstream asserts it should have fired.
- **Pin `--advanced` and `--iterate-rules` in CI**, the way you pin a dependency.
- Exit codes separate *violations* (1) from *could not validate* (2). A job that
  treats them alike will eventually go green on rules that never ran.
- The **WebAssembly build has no rules at all** — browser and editor validation
  is validation-only.

Next: [3 — Release and change](03-release-and-change.ipynb).